# 02 — Conjugate Models
**References:** Gelman et al. BDA3 Ch. 2–3 · Robert (2007) Ch. 3

## Narrative thread
```
Exponential family -> Conjugacy theorem -> Beta-Binomial -> Normal-Normal -> Gamma-Poisson -> Normal-Inverse-Gamma -> Dirichlet-Multinomial
```

## 1. Exponential family and natural conjugates

A distribution belongs to the **exponential family** if its density has the form:
$$f(x \mid \boldsymbol{\eta}) = h(x)\, \exp\!\left(\boldsymbol{\eta}^\top T(x) - A(\boldsymbol{\eta})\right)$$

where $\boldsymbol{\eta}$ are the **natural parameters**, $T(x)$ the **sufficient statistics**, and $A(\boldsymbol{\eta}) = \log \int h(x)e^{\boldsymbol{\eta}^\top T(x)}dx$ the **log-partition function**.

**Conjugacy theorem:** The natural conjugate prior for an exponential family likelihood is:
$$p(\boldsymbol{\eta} \mid \boldsymbol{\chi}, \nu) \propto \exp\!\left(\boldsymbol{\eta}^\top \boldsymbol{\chi} - \nu A(\boldsymbol{\eta})\right)$$

After observing $n$ i.i.d. data points, the posterior is the same family with updated hyperparameters:
$$\boldsymbol{\chi} \to \boldsymbol{\chi} + \sum_{i=1}^n T(x_i), \quad \nu \to \nu + n$$

> Conjugate priors are "fake data" — $\boldsymbol{\chi}$ and $\nu$ act like pseudo-observations. The prior is equivalent to having seen $\nu$ previous observations with sufficient statistic $\boldsymbol{\chi}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gammaln

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Conjugate family zoo ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
theta_grid = np.linspace(0.001, 0.999, 400)

# 1. Beta-Binomial
ax = axes[0, 0]
a0, b0 = 2, 5       # prior Beta(2,5)
x_obs, n_obs = 7, 10  # data: 7 successes in 10 trials
a_post, b_post = a0 + x_obs, b0 + n_obs - x_obs
ax.plot(theta_grid, stats.beta.pdf(theta_grid, a0, b0),     color='#06d6a0', lw=2, label=f'Prior Beta({a0},{b0})')
ax.plot(theta_grid, stats.beta.pdf(theta_grid, a_post, b_post), color='#4361ee', lw=2.5, label=f'Post Beta({a_post},{b_post})')
ax.axvline(x_obs/n_obs, color='#f72585', lw=1.5, linestyle='--', label=f'MLE={x_obs/n_obs}')
ax.axvline(a_post/(a_post+b_post), color='#4361ee', lw=1.5, linestyle=':', label=f'Post mean={a_post/(a_post+b_post):.2f}')
ax.set_title('Beta-Binomial\n$p\\sim\\text{Beta}(\\alpha_0+x, \\beta_0+n-x)$')
ax.set_xlabel('p'); ax.legend(fontsize=7)

# 2. Gamma-Poisson
ax = axes[0, 1]
lam_grid = np.linspace(0, 12, 400)
a0_p, b0_p = 2, 1     # prior Gamma(2,1)
data_pois = np.array([3, 1, 4, 2, 5, 3, 2])  # Poisson data
a_post_p  = a0_p + data_pois.sum()
b_post_p  = b0_p + len(data_pois)
ax.plot(lam_grid, stats.gamma.pdf(lam_grid, a0_p, scale=1/b0_p),   color='#06d6a0', lw=2, label=f'Prior Gamma({a0_p},{b0_p})')
ax.plot(lam_grid, stats.gamma.pdf(lam_grid, a_post_p, scale=1/b_post_p), color='#4361ee', lw=2.5, label=f'Post Gamma({a_post_p},{b_post_p})')
ax.axvline(data_pois.mean(), color='#f72585', lw=1.5, linestyle='--', label=f'MLE $\\bar x$={data_pois.mean():.2f}')
ax.set_title('Gamma-Poisson\n$\\lambda\\sim\\text{Gamma}(\\alpha_0+\\sum x_i, \\beta_0+n)$')
ax.set_xlabel('$\\lambda$'); ax.legend(fontsize=7)

# 3. Gamma-Exponential (rate lambda)
ax = axes[0, 2]
a0_e, b0_e = 3, 1     # prior Gamma(3,1)
data_exp = np.random.exponential(scale=0.5, size=10)  # Exp(lambda=2) data
a_post_e  = a0_e + len(data_exp)
b_post_e  = b0_e + data_exp.sum()
lam_g = np.linspace(0, 8, 400)
ax.plot(lam_g, stats.gamma.pdf(lam_g, a0_e, scale=1/b0_e),   color='#06d6a0', lw=2, label=f'Prior Gamma({a0_e},{b0_e})')
ax.plot(lam_g, stats.gamma.pdf(lam_g, a_post_e, scale=1/b_post_e), color='#4361ee', lw=2.5,
        label=f'Post Gamma({a_post_e},{b_post_e:.1f})')
ax.axvline(len(data_exp)/data_exp.sum(), color='#f72585', lw=1.5, linestyle='--', label=f'MLE $n/\\sum x$={len(data_exp)/data_exp.sum():.2f}')
ax.set_title('Gamma-Exponential\n$\\lambda\\sim\\text{Gamma}(\\alpha_0+n, \\beta_0+\\sum x_i)$')
ax.set_xlabel('$\\lambda$'); ax.legend(fontsize=7)

# 4. Normal-Normal (known sigma)
ax = axes[1, 0]
mu_g = np.linspace(-4, 6, 400)
mu0_n, tau0_n, sigma_n = 0.0, 2.0, 1.0
data_norm = np.array([2.1, 1.9, 2.5, 1.8, 2.3])
n_n = len(data_norm); xbar_n = data_norm.mean()
tau_n_inv2 = 1/tau0_n**2 + n_n/sigma_n**2
mu_n_n = (mu0_n/tau0_n**2 + n_n*xbar_n/sigma_n**2) / tau_n_inv2
tau_n_n = 1/np.sqrt(tau_n_inv2)
ax.plot(mu_g, stats.norm.pdf(mu_g, mu0_n, tau0_n), color='#06d6a0', lw=2, label=f'Prior $\\mathcal{{N}}({mu0_n},{tau0_n}^2)$')
ax.plot(mu_g, stats.norm.pdf(mu_g, mu_n_n, tau_n_n), color='#4361ee', lw=2.5, label=f'Post $\\mathcal{{N}}({mu_n_n:.2f},{tau_n_n:.2f}^2)$')
ax.axvline(xbar_n, color='#f72585', lw=1.5, linestyle='--', label=f'MLE $\\bar x$={xbar_n:.2f}')
ax.set_title('Normal-Normal (known $\\sigma$)\nPrecision-weighted average')
ax.set_xlabel('$\\mu$'); ax.legend(fontsize=7)

# 5. Inverse-Gamma for Normal variance (known mu)
ax = axes[1, 1]
sig2_g = np.linspace(0.01, 6, 400)
a0_ig, b0_ig = 2, 2      # prior IG(2,2)
mu_known = 0.0
data_var = np.random.normal(mu_known, 1.5, 8)
n_v = len(data_var)
a_post_ig = a0_ig + n_v/2
b_post_ig = b0_ig + 0.5 * np.sum((data_var - mu_known)**2)
ig_prior = stats.invgamma.pdf(sig2_g, a0_ig, scale=b0_ig)
ig_post  = stats.invgamma.pdf(sig2_g, a_post_ig, scale=b_post_ig)
ax.plot(sig2_g, ig_prior, color='#06d6a0', lw=2, label=f'Prior IG({a0_ig},{b0_ig})')
ax.plot(sig2_g, ig_post,  color='#4361ee', lw=2.5, label=f'Post IG({a_post_ig:.1f},{b_post_ig:.1f})')
ax.axvline(np.var(data_var), color='#f72585', lw=1.5, linestyle='--', label=f'MLE $s^2$={np.var(data_var):.2f}')
ax.set_title('Inverse-Gamma for Normal variance\n$\\sigma^2\\sim\\text{IG}(\\alpha_0+n/2, \\beta_0+\\text{SS}/2)$')
ax.set_xlabel('$\\sigma^2$'); ax.legend(fontsize=7)

# 6. Dirichlet-Multinomial
ax = axes[1, 2]
# Project onto simplex: show marginal Beta distributions
alpha0_dir = np.array([1, 1, 1])  # symmetric Dirichlet
counts = np.array([15, 8, 12])    # observed counts
alpha_post_dir = alpha0_dir + counts
theta_g = np.linspace(0, 1, 400)
colors_dir = ['#4361ee', '#f72585', '#06d6a0']
for k, (a0k, apk, c) in enumerate(zip(alpha0_dir, alpha_post_dir, colors_dir)):
    ax.plot(theta_g, stats.beta.pdf(theta_g, a0k, alpha0_dir.sum()-a0k), color=c, lw=1.5, linestyle='--')
    ax.plot(theta_g, stats.beta.pdf(theta_g, apk, alpha_post_dir.sum()-apk), color=c, lw=2.5, label=f'Cat {k+1}: post mean={apk/alpha_post_dir.sum():.2f}')
ax.set_title('Dirichlet-Multinomial\nMarginals of Dir posterior are Beta')
ax.set_xlabel('$p_k$'); ax.legend(fontsize=7)

plt.suptitle('Conjugate Bayesian Models — Prior and Posterior', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. The Normal-Inverse-Gamma model (both $\mu$ and $\sigma^2$ unknown)

$$\mu \mid \sigma^2 \sim \mathcal{N}(\mu_0,\, \sigma^2/\kappa_0), \qquad \sigma^2 \sim \text{IG}(\alpha_0, \beta_0)$$

Joint prior: Normal-Inverse-Gamma (NIG). After observing $\mathbf{x}$:

$$\kappa_n = \kappa_0 + n, \quad \mu_n = \frac{\kappa_0 \mu_0 + n\bar x}{\kappa_n}$$
$$\alpha_n = \alpha_0 + n/2, \quad \beta_n = \beta_0 + \frac{1}{2}\sum(x_i - \bar x)^2 + \frac{\kappa_0 n (\bar x - \mu_0)^2}{2\kappa_n}$$

**Marginal posterior of $\mu$** (marginalizing over $\sigma^2$) is a Student-t:
$$\mu \mid \mathbf{x} \sim t_{2\alpha_n}\!\left(\mu_n,\, \beta_n/(\alpha_n \kappa_n)\right)$$

In [ ]:
# ── Normal-Inverse-Gamma: posterior for mu and sigma^2 jointly ────────────
np.random.seed(42)
data_nig = np.random.normal(3, 1.5, size=20)

# Prior hyperparameters
mu0_nig, kappa0, alpha0_nig, beta0_nig = 0.0, 1.0, 2.0, 2.0

# Posterior hyperparameters
n_nig  = len(data_nig); xbar_nig = data_nig.mean()
kappa_n   = kappa0 + n_nig
mu_n_nig  = (kappa0*mu0_nig + n_nig*xbar_nig) / kappa_n
alpha_n_nig = alpha0_nig + n_nig/2
beta_n_nig  = beta0_nig + 0.5*np.sum((data_nig - xbar_nig)**2) + \
              kappa0*n_nig*(xbar_nig - mu0_nig)**2 / (2*kappa_n)

# Marginal posterior of mu: Student-t
df_mu  = 2*alpha_n_nig
scale_mu = np.sqrt(beta_n_nig / (alpha_n_nig * kappa_n))

# Marginal posterior of sigma^2: Inverse-Gamma
# Mean of IG(alpha_n, beta_n) = beta_n/(alpha_n - 1)
post_mean_sigma2 = beta_n_nig / (alpha_n_nig - 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Posterior of mu
mu_range = np.linspace(-2, 7, 400)
post_mu_pdf = stats.t.pdf(mu_range, df=df_mu, loc=mu_n_nig, scale=scale_mu)
axes[0].plot(mu_range, post_mu_pdf, color='#4361ee', lw=2.5, label=f'Post $\\mu \\sim t_{{{df_mu:.0f}}}({mu_n_nig:.2f})$')
axes[0].axvline(xbar_nig, color='#f72585', lw=2, linestyle='--', label=f'$\\bar x={xbar_nig:.2f}$')
ci = stats.t.ppf([0.025, 0.975], df=df_mu, loc=mu_n_nig, scale=scale_mu)
axes[0].fill_between(mu_range, post_mu_pdf, where=(mu_range>=ci[0])&(mu_range<=ci[1]),
                     alpha=0.2, color='#4361ee', label=f'95% CI [{ci[0]:.2f},{ci[1]:.2f}]')
axes[0].set_xlabel('$\\mu$'); axes[0].set_title('Posterior of $\\mu$ (marginalized over $\\sigma^2$)\nStudent-t; wider than Normal due to $\\sigma^2$ uncertainty')
axes[0].legend(fontsize=8)

# Posterior of sigma^2
sig2_range = np.linspace(0.01, 8, 400)
post_sig2_pdf = stats.invgamma.pdf(sig2_range, alpha_n_nig, scale=beta_n_nig)
axes[1].plot(sig2_range, post_sig2_pdf, color='#4361ee', lw=2.5,
             label=f'Post $\\sigma^2 \\sim \\text{{IG}}({alpha_n_nig:.1f},{beta_n_nig:.1f})$')
axes[1].axvline(np.var(data_nig, ddof=1), color='#f72585', lw=2, linestyle='--', label=f'$s^2={np.var(data_nig,ddof=1):.2f}$')
axes[1].axvline(post_mean_sigma2, color='#06d6a0', lw=2, label=f'Post mean={post_mean_sigma2:.2f}')
axes[1].set_xlabel('$\\sigma^2$'); axes[1].set_title('Posterior of $\\sigma^2$ (marginalized over $\\mu$)\nInverse-Gamma')
axes[1].legend(fontsize=8)

plt.suptitle('Normal-Inverse-Gamma: Marginal Posteriors', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('=== Normal-Inverse-Gamma posterior ===')
print(f'  n={n_nig},  x_bar={xbar_nig:.3f},  s^2={np.var(data_nig,ddof=1):.3f}')
print(f'  Posterior mu: t({df_mu:.0f}, loc={mu_n_nig:.3f}, scale={scale_mu:.3f})')
print(f'  Posterior sigma^2: IG(alpha={alpha_n_nig:.1f}, beta={beta_n_nig:.1f})')
print(f'  Post. mean sigma^2 = {post_mean_sigma2:.3f}')

## 3. Summary table

| Likelihood | Parameter | Conjugate prior | Posterior update |
|---|---|---|---|
| Bernoulli/Binomial | $p$ | Beta$(\alpha,\beta)$ | Beta$(\alpha+x, \beta+n-x)$ |
| Poisson | $\lambda$ | Gamma$(\alpha,\beta)$ | Gamma$(\alpha+\sum x, \beta+n)$ |
| Exponential | $\lambda$ | Gamma$(\alpha,\beta)$ | Gamma$(\alpha+n, \beta+\sum x)$ |
| Normal (known $\sigma^2$) | $\mu$ | Normal$(\mu_0,\tau_0^2)$ | Normal (precision-weighted) |
| Normal (known $\mu$) | $\sigma^2$ | IG$(\alpha,\beta)$ | IG$(\alpha+n/2, \beta+\text{SS}/2)$ |
| Normal (both unknown) | $(\mu,\sigma^2)$ | NIG | NIG with updated params |
| Multinomial | $\mathbf{p}$ | Dirichlet$(\boldsymbol{\alpha})$ | Dirichlet$(\boldsymbol{\alpha}+\mathbf{x})$ |

**Next:** notebook 03 — prior distributions: how to choose, non-informative priors, Jeffreys prior, weakly informative priors.